Importing packages

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    mean_squared_error,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    make_scorer,
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.compose import TransformedTargetRegressor

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt
import joblib
from datetime import datetime

In [9]:
# df = pd.read_csv("../data_model/db_additional.csv")
df = pd.read_csv("/kaggle/input/ml-prep/db_additional_fe.csv")

Regressor - param search

In [10]:
X = df.drop(columns='volume_total')
y = df['volume_total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"y_train range: [{y_train.min():.0f}, {y_train.max():.0f}]")

Train: (78341, 1555), Test: (19586, 1555)
y_train range: [0, 1615812275]


In [11]:
(df['volume_total'] > 0).sum()

np.int64(97927)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026)

In [13]:
RANDOM_STATE = 2026
N_JOBS = -1
N_TRIALS = 50
CV_FOLDS = 3


models = {
    "xgb": XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE
    ),
    
    "lgbm": LGBMRegressor(
        objective="regression",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE
    ),
    
    "catboost": CatBoostRegressor(
        loss_function="RMSE",
        verbose=0,
        random_state=RANDOM_STATE
    ),
    
    "rf": RandomForestRegressor(
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE
    ),
    
    "xgb_log": TransformedTargetRegressor(
        regressor=XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
}

param_spaces = {
    "xgb": {
        "n_estimators": ("int", 200, 1200),
        "max_depth": ("int", 3, 12),
        "learning_rate": ("float", 0.01, 0.1),
        "subsample": ("float", 0.6, 1.0),
        "colsample_bytree": ("float", 0.6, 1.0),
        "min_child_weight": ("int", 1, 10),
        "gamma": ("float", 0, 2.0),
        "reg_alpha": ("float", 0, 1.0),
        "reg_lambda": ("float", 0, 1.0),
    },
    
    "lgbm": {
        "n_estimators": ("int", 200, 1200),
        "max_depth": ("int", -1, 12),
        "learning_rate": ("float", 0.01, 0.1),
        "num_leaves": ("int", 16, 256),
        "subsample": ("float", 0.6, 1.0),
        "colsample_bytree": ("float", 0.6, 1.0),
        "min_child_samples": ("int", 5, 50),
        "reg_alpha": ("float", 0, 1.0),
        "reg_lambda": ("float", 0, 1.0),
    },
    
    "catboost": {
        "depth": ("int", 4, 10),
        "learning_rate": ("float", 0.01, 0.1),
        "n_estimators": ("int", 200, 1200),
        "l2_leaf_reg": ("float", 1, 10),
        "bagging_temperature": ("float", 0, 5),
    },
    
    "rf": {
        "n_estimators": ("int", 200, 800),
        "max_depth": ("int", 3, 20),
        "min_samples_split": ("int", 2, 20),
        "min_samples_leaf": ("int", 1, 10),
        "max_features": ("categorical", ["sqrt", "log2", 0.5, 0.8]),
    },
    
    "xgb_log": {  # Same as xgb
        "n_estimators": ("int", 200, 1200),
        "max_depth": ("int", 3, 12),
        "learning_rate": ("float", 0.01, 0.1),
        "subsample": ("float", 0.6, 1.0),
        "colsample_bytree": ("float", 0.6, 1.0),
        "min_child_weight": ("int", 1, 10),
        "gamma": ("float", 0, 2.0),
        "reg_alpha": ("float", 0, 1.0),
        "reg_lambda": ("float", 0, 1.0),
    },
}

model_constructors = {
    "xgb": lambda params: XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        **params
    ),
    
    "lgbm": lambda params: LGBMRegressor(
        objective="regression",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        **params
    ),
    
    "catboost": lambda params: CatBoostRegressor(
        loss_function="RMSE",
        verbose=0,
        random_state=RANDOM_STATE,
        **params
    ),
    
    "rf": lambda params: RandomForestRegressor(
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        **params
    ),
    
    "xgb_log": lambda params: TransformedTargetRegressor(
        regressor=XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            **params
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
}


def create_objective(model_name, X_train, y_train, cv=CV_FOLDS):
    """Create an Optuna objective function for hyperparameter optimization."""
    
    def objective(trial):
        params = {}
        for param_name, param_config in param_spaces[model_name].items():
            param_type = param_config[0]
            
            if param_type == "int":
                params[param_name] = trial.suggest_int(param_name, param_config[1], param_config[2])
            elif param_type == "float":
                params[param_name] = trial.suggest_float(param_name, param_config[1], param_config[2])
            elif param_type == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_config[1])
        
        model = model_constructors[model_name](params)
        
        scores = cross_val_score(
            model, X_train, y_train,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=N_JOBS
        )
        
        return -scores.mean()  # Return negative RMSE (Optuna minimizes)
    
    return objective


In [14]:
results = {}

for name, estimator in models.items():
    print(f"\n{'='*60}")
    print(f"{'='*20} {name.upper()} {'='*20}")
    print(f"{'='*60}")
    
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{name}_regression"
    )
    objective = create_objective(name, X_train, y_train, cv=CV_FOLDS)
    
    print(f"Running {N_TRIALS} trials...")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    
    best_params = study.best_params
    best_model = model_constructors[name](best_params)
    
    best_model.fit(X_train, y_train)
    
    y_pred = best_model.predict(X_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    
    results[name] = {
        "study": study,
        "best_model": best_model,
        "best_params": best_params,
        "cv_best_score": study.best_value,
        "test_rmse": test_rmse,
    }
    
    print(f"\n✓ Optimization complete!")
    print(f"  Best params: {best_params}")
    print(f"  CV RMSE: {study.best_value:.4f}")
    print(f"  Test RMSE: {test_rmse:.4f}")



print(f"\n{'='*60}")
print(f"{'='*20} SUMMARY {'='*20}")
print(f"{'='*60}\n")

print(f"{'Model':<12} {'CV RMSE':<12} {'Test RMSE':<12} {'Difference':<12}")
print("-" * 60)

for name, result in results.items():
    cv_score = result['cv_best_score']
    test_score = result['test_rmse']
    diff = test_score - cv_score
    print(f"{name:<12} {cv_score:<12.4f} {test_score:<12.4f} {diff:+12.4f}")

best_model_name = min(results.items(), key=lambda x: x[1]['test_rmse'])[0]
print(f"\n✓ Best model: {best_model_name.upper()} (Test RMSE: {results[best_model_name]['test_rmse']:.4f})")

[I 2026-01-26 21:34:04,033] A new study created in memory with name: xgb_regression



==================== XGB ====================
Running 50 trials...


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-01-26 21:34:14,791] Trial 0 failed with parameters: {'n_estimators': 419, 'max_depth': 7, 'learning_rate': 0.09789719303442711, 'subsample': 0.6355596086592517, 'colsample_bytree': 0.7917159438720727, 'min_child_weight': 10, 'gamma': 0.3962821780378363, 'reg_alpha': 0.9117151818452747, 'reg_lambda': 0.5582501148658583} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_101/1586272401.py", line 158, in objective
    scores = cross_val_score(
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 684, in cross_val_score

KeyboardInterrupt: 

In [ ]:
best_name = min(results, key=lambda k: results[k]["test_rmse"])
best_model = results[best_name]["best_model"]

print(f"\n{'='*60}")
print(f"Najlepszy model: {best_name.upper()}")
print(f"{'='*60}")
print(f"Test RMSE: {results[best_name]['test_rmse']:.2f}")
print(f"\nNajlepsze parametry:")
for param, value in results[best_name]['best_params'].items():
    print(f"  {param}: {value}")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

best_name = min(results, key=lambda k: results[k]["test_rmse"])
best_model = results[best_name]["best_model"]

model_filename = f"best_regressor_{best_name}_{timestamp}.pkl"
joblib.dump(best_model, model_filename)
print(f"\nNajlepszy model zapisany: {model_filename}")

Classifier - param search

In [ ]:
y = (df["volume_total"] > 10_000).astype(int)
X = df.drop(columns=["volume_total"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026, stratify=y
)

train_class_dist = y_train.value_counts(normalize=True)
test_class_dist = y_test.value_counts(normalize=True)

In [ ]:
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nRozkład klas (train):")
print(f"  0 (<=10k): {train_class_dist[0]:.1%}")
print(f"  1 (>10k):  {train_class_dist[1]:.1%}")
print(f"\nRozkład klas (test):")
print(f"  0 (<=10k): {test_class_dist[0]:.1%}")
print(f"  1 (>10k):  {test_class_dist[1]:.1%}")

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\npos_weight: {pos_weight:.2f}")

In [ ]:
RANDOM_STATE = 2026
N_JOBS = -1
N_TRIALS = 50
CV_FOLDS = 3


models_sampling = {
    "xgb_smote_under": {
        "samplers": [
            ("smote", SMOTE(sampling_strategy=0.3, random_state=RANDOM_STATE)),
            ("under", RandomUnderSampler(sampling_strategy=0.7, random_state=RANDOM_STATE)),
        ],
        "classifier": XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="aucpr",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            scale_pos_weight=pos_weight,
        ),
    },
    
    "lgbm_smote_over": {
        "samplers": [
            ("smote", SMOTE(sampling_strategy=0.3, random_state=RANDOM_STATE)),
            ("over", RandomOverSampler(sampling_strategy=0.5, random_state=RANDOM_STATE)),
        ],
        "classifier": LGBMClassifier(
            objective="binary",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            is_unbalance=True,
        ),
    },
    
    "cat_smotetomek": {
        "samplers": [("smotetomek", SMOTETomek(random_state=RANDOM_STATE))],
        "classifier": CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="Recall",
            verbose=0,
            random_seed=RANDOM_STATE,
            scale_pos_weight=pos_weight,
        ),
    },
    
    "rf_smoteenn": {
        "samplers": [("smoteenn", SMOTEENN(random_state=RANDOM_STATE))],
        "classifier": RandomForestClassifier(
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            class_weight="balanced_subsample",
        ),
    },
}

param_spaces = {
    "xgb": {
        "clf__n_estimators": ("int", 200, 1000),
        "clf__max_depth": ("int", 3, 10),
        "clf__learning_rate": ("float", 0.01, 0.1),
        "clf__subsample": ("float", 0.6, 1.0),
        "clf__colsample_bytree": ("float", 0.6, 1.0),
        "clf__min_child_weight": ("int", 1, 10),
        "clf__gamma": ("float", 0.0, 2.0),
        "clf__reg_alpha": ("float", 0.0, 1.0),
        "clf__reg_lambda": ("float", 0.0, 1.0),
    },
    
    "lgbm": {
        "clf__n_estimators": ("int", 200, 1000),
        "clf__max_depth": ("int", -1, 10),
        "clf__learning_rate": ("float", 0.01, 0.1),
        "clf__num_leaves": ("int", 16, 256),
        "clf__subsample": ("float", 0.6, 1.0),
        "clf__colsample_bytree": ("float", 0.6, 1.0),
        "clf__min_child_samples": ("int", 10, 60),
        "clf__reg_alpha": ("float", 0.0, 1.0),
        "clf__reg_lambda": ("float", 0.0, 1.0),
    },
    
    "cat": {
        "clf__depth": ("int", 4, 10),
        "clf__learning_rate": ("float", 0.01, 0.1),
        "clf__n_estimators": ("int", 200, 1000),
        "clf__l2_leaf_reg": ("float", 1.0, 10.0),
    },
    
    "rf": {
        "clf__n_estimators": ("int", 200, 800),
        "clf__max_depth": ("int", 3, 20),
        "clf__min_samples_split": ("int", 2, 20),
        "clf__min_samples_leaf": ("int", 1, 10),
        "clf__max_features": ("categorical", ["sqrt", "log2", 0.5]),
    },
}


classifier_constructors = {
    "xgb": lambda: XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="aucpr",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        scale_pos_weight=pos_weight,
    ),
    
    "lgbm": lambda: LGBMClassifier(
        objective="binary",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        is_unbalance=True,
    ),
    
    "cat": lambda: CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Recall",
        verbose=0,
        random_seed=RANDOM_STATE,
        scale_pos_weight=pos_weight,
    ),
    
    "rf": lambda: RandomForestClassifier(
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
    ),
}


def get_model_type(name):
    """Extract model type from pipeline name."""
    if "xgb" in name:
        return "xgb"
    elif "lgbm" in name:
        return "lgbm"
    elif "cat" in name:
        return "cat"
    elif "rf" in name:
        return "rf"
    return None


def create_classification_objective(name, samplers, X_train, y_train, cv):
    """Create an Optuna objective function for classification optimization."""
    
    model_type = get_model_type(name)
    
    def objective(trial):
        params = {}
        for param_name, param_config in param_spaces[model_type].items():
            param_type = param_config[0]
            
            if param_type == "int":
                params[param_name] = trial.suggest_int(param_name, param_config[1], param_config[2])
            elif param_type == "float":
                params[param_name] = trial.suggest_float(param_name, param_config[1], param_config[2])
            elif param_type == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_config[1])
        
        clf = classifier_constructors[model_type]()
        
        steps = samplers + [("clf", clf)]
        pipe = Pipeline(steps=steps)


        pipe.set_params(**params)
        
        scores = cross_val_score(
            pipe, X_train, y_train,
            cv=cv,
            scoring="recall",
            n_jobs=N_JOBS
        )
        return scores.mean()  # Return recall (Optuna maximizes)
    
    return objective


In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results_sampling = {}


for name, config in models_sampling.items():
    print(f"\n{'='*60}")
    print(f"{'='*20} {name.upper()} {'='*20}")
    print(f"{'='*60}")
    
    samplers = config["samplers"]
    model_type = get_model_type(name)
    
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{name}_classification"
    )
    
    objective = create_classification_objective(name, samplers, X_train, y_train, cv)
    
    print(f"Running {N_TRIALS} trials...")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    
    best_params = study.best_params
    
    clf = classifier_constructors[model_type]()
    clf_params = {k.replace("clf__", ""): v for k, v in best_params.items()}
    clf.set_params(**clf_params)
    
    steps = samplers + [("clf", clf)]
    best_pipe = Pipeline(steps=steps)
    
    best_pipe.fit(X_train, y_train)
    
    y_pred = best_pipe.predict(X_test)
    y_pred_proba = best_pipe.predict_proba(X_test)[:, 1]
    
    results_sampling[name] = {
        "study": study,
        "best_pipe": best_pipe,
        "best_params": best_params,
        "cv_recall": study.best_value,
        "test_recall": recall_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_pred_proba),
    }
    
    print(f"\n✓ Optimization complete!")
    print(f"  Best params: {best_params}")
    print(f"  CV Recall: {study.best_value:.4f}")
    print(f"  Test Recall: {results_sampling[name]['test_recall']:.4f}")
    print(f"  Test Precision: {results_sampling[name]['test_precision']:.4f}")
    print(f"  Test F1: {results_sampling[name]['test_f1']:.4f}")
    print(f"  Test AUC: {results_sampling[name]['test_auc']:.4f}")
    
    print(f"\n{classification_report(y_test, y_pred, target_names=['<=10k', '>10k'])}")



print(f"\n{'='*60}")
print(f"{'='*20} SUMMARY {'='*20}")
print(f"{'='*60}\n")

print(f"{'Model':<20} {'CV Recall':<12} {'Test Recall':<12} {'Precision':<12} {'F1':<12} {'AUC':<12}")
print("-" * 90)

for name, result in results_sampling.items():
    cv_recall = result['cv_recall']
    test_recall = result['test_recall']
    precision = result['test_precision']
    f1 = result['test_f1']
    auc = result['test_auc']
    print(f"{name:<20} {cv_recall:<12.4f} {test_recall:<12.4f} {precision:<12.4f} {f1:<12.4f} {auc:<12.4f}")

# Find best model based on test recall
best_model_name = max(results_sampling.items(), key=lambda x: x[1]['test_recall'])[0]
print(f"\n✓ Best model by Recall: {best_model_name.upper()} (Recall: {results_sampling[best_model_name]['test_recall']:.4f})")

# Find best model based on F1
best_f1_name = max(results_sampling.items(), key=lambda x: x[1]['test_f1'])[0]
print(f"✓ Best model by F1: {best_f1_name.upper()} (F1: {results_sampling[best_f1_name]['test_f1']:.4f})")

In [ ]:
best_name = max(results_sampling, key=lambda k: results_sampling[k]["test_recall"])
best_pipe = results_sampling[best_name]["best_pipe"]

print(f"\n{'='*60}")
print(f"NAJLEPSZY MODEL: {best_name.upper()}")
print(f"{'='*60}")
print(f"Test Recall:    {results_sampling[best_name]['test_recall']:.4f}")
print(f"Test Precision: {results_sampling[best_name]['test_precision']:.4f}")
print(f"Test F1:        {results_sampling[best_name]['test_f1']:.4f}")
print(f"Test AUC:       {results_sampling[best_name]['test_auc']:.4f}")

y_pred_best = best_pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

print(f"\n{'='*60}")
print("Confusion matrix (najlepszy model):")
print(f"{'='*60}")
print(cm)
print(f"\nTrue Negatives:  {cm[0,0]:>6d}")
print(f"False Positives: {cm[0,1]:>6d}")
print(f"False Negatives: {cm[1,0]:>6d}  (Błąd II rodzaju)")
print(f"True Positives:  {cm[1,1]:>6d}")

fnr = cm[1,0] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0
print(f"\nFalse Negative Rate (FNR): {fnr:.4f}")

In [ ]:
y_pred_proba_best = best_pipe.predict_proba(X_test)[:, 1]

thresholds = [0.2, 0.3, 0.4, 0.5]

print(f"{'='*80}")
print("Analiza progów decyzyjnych (minimalizacja FN)")
print(f"{'='*80}")
print(f"{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'F1':<10} {'FN':<8} {'FP':<8}")
print(f"{'-'*80}")

for threshold in thresholds:
    y_pred_thresh = (y_pred_proba_best >= threshold).astype(int)
    cm_thresh = confusion_matrix(y_test, y_pred_thresh)
    
    recall_t = recall_score(y_test, y_pred_thresh)
    precision_t = precision_score(y_test, y_pred_thresh)
    f1_t = f1_score(y_test, y_pred_thresh)
    fn_t = cm_thresh[1, 0]
    fp_t = cm_thresh[0, 1]
    
    print(f"{threshold:<12.2f} {recall_t:<10.4f} {precision_t:<12.4f} {f1_t:<10.4f} {fn_t:<8d} {fp_t:<8d}")


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

best_name = max(results_sampling, key=lambda k: results_sampling[k]["test_recall"])
best_pipe = results_sampling[best_name]["best_pipe"]

pipe_filename = f"best_classifier_{best_name}_{timestamp}.pkl"
joblib.dump(best_pipe, pipe_filename)
print(f"\nNajlepszy pipeline zapisany: {pipe_filename}")